In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb

RANDOM_STATE = 42

train_trips = pd.read_csv('public_trip_data.csv')
train_events = pd.read_csv('public_trip_event_log.csv')
train_attrs = pd.read_csv('public_trip_event_attributes.csv')

test_trips = pd.read_csv('private_trip_data.csv')
test_events = pd.read_csv('private_trip_event_log.csv')
test_attrs = pd.read_csv('private_trip_event_attributes.csv')

print('Train trips:', train_trips.shape)
print('Test trips:', test_trips.shape)

Train trips: (65289, 20)
Test trips: (21764, 13)


# Feature Engineering

In [12]:
def engineer_event_features(events):
    events = events.copy()
    events['EventTimestamp'] = pd.to_datetime(events['EventTimestamp'])

    num_events = events.groupby('TripID').size().rename('NumEvents')

    key_names = ['Submit Travel Request', 'Travel Request Approved',
                 'Take Departure Flight', 'Take Departure Train', 'Pickup Rental']
    key = events[events['EventName'].isin(key_names)]
    piv = key.pivot_table(index='TripID', columns='EventName', values='EventTimestamp', aggfunc='first')
    for col in key_names:
        if col not in piv.columns:
            piv[col] = pd.NaT

    piv['DepartureTime'] = piv[['Take Departure Flight', 'Take Departure Train', 'Pickup Rental']].min(axis=1)
    piv['LeadTimeDays'] = (piv['DepartureTime'] - piv['Submit Travel Request']).dt.total_seconds() / 86400
    piv['ApprovalLeadDays'] = (piv['Travel Request Approved'] - piv['Submit Travel Request']).dt.total_seconds() / 86400

    flag_events = {
        'HasManagerPreapproved': 'Manager Preapproved',
        'HasTripExtension': 'Trip Extension',
        'HasFlightChange': 'Flight Change',
        'HasFlightCancellation': 'Flight Cancellation',
        'HasFlightDelay': 'Flight Delay',
        'HasMissedFlight': 'Missed Flight',
        'HasHotelChange': 'Hotel Change',
        'HasModeChange': 'Mode of Transportation Change',
        'HasItineraryEdit': 'Itinerary Edit',
    }
    flags = pd.DataFrame(index=piv.index)
    for colname, evname in flag_events.items():
        trips_with_event = set(events.loc[events['EventName'] == evname, 'TripID'])
        flags[colname] = piv.index.isin(trips_with_event).astype(int)

    out = piv[['LeadTimeDays', 'ApprovalLeadDays']].join(flags).join(num_events)
    out['LeadTimeMissing'] = out['LeadTimeDays'].isna().astype(int)
    return out.reset_index()


def engineer_attribute_features(attrs):
    keep = attrs[['TripID', 'DaysPreapproved', 'ExtensionLength']].copy()
    keep['DaysPreapproved'] = keep['DaysPreapproved'].fillna(0)
    keep['ExtensionLength'] = keep['ExtensionLength'].fillna(0)
    return keep


train_ev_feats = engineer_event_features(train_events)
train_at_feats = engineer_attribute_features(train_attrs)
train_df = train_trips.merge(train_ev_feats, on='TripID', how='left').merge(train_at_feats, on='TripID', how='left')

test_ev_feats = engineer_event_features(test_events)
test_at_feats = engineer_attribute_features(test_attrs)
test_df = test_trips.merge(test_ev_feats, on='TripID', how='left').merge(test_at_feats, on='TripID', how='left')

print(train_df.shape, test_df.shape)
train_df.head(3)

(65289, 35) (21764, 28)


,TripID,DepartureLocationCountry,DepartureLocationCity,ArrivalLocationCountry,ArrivalLocationCity,ShippingType,ShippingTypeDescription,Purpose,OutOfPolicy,EntitiyCode,...,HasFlightCancellation,HasFlightDelay,HasMissedFlight,HasHotelChange,HasModeChange,HasItineraryEdit,NumEvents,LeadTimeMissing,DaysPreapproved,ExtensionLength
0,1,CN,Beijing,IN,New Delhi,12,Business Class Flight,Customer Visit,No,9000,...,0,0,0,0,0,0,10,0,2,0.0
1,3,US,New York,MX,Mexico City,11,First Class Flight,Customer Visit,Yes,6000,...,0,0,0,0,0,0,10,0,2,0.0
2,6,BR,SÃ£o Paulo,ZA,Johannesburg,10,Economy Flight,Conference/Exhibition,No,9000,...,0,0,0,0,0,0,10,0,1,0.0


In [13]:
LEAKAGE_COLS = ['Departure_CO2e', 'Return_CO2e', 'Hotel_CO2e', 'Spend_CO2e', 'TotalCO2e', 'HighCarbon']
ID_COLS = ['TripID', 'EmployeeNumber']

CATEGORICAL_COLS = [
    'DepartureLocationCountry', 'DepartureLocationCity', 'ArrivalLocationCountry', 'ArrivalLocationCity',
    'ShippingTypeDescription', 'Purpose', 'OutOfPolicy', 'BusinessUnit'
]
NUMERIC_COLS = [
    'ShippingType', 'EntitiyCode', 'HotelNights', 'NetCosts',
    'LeadTimeDays', 'ApprovalLeadDays', 'NumEvents', 'LeadTimeMissing',
    'HasManagerPreapproved', 'HasTripExtension', 'HasFlightChange', 'HasFlightCancellation',
    'HasFlightDelay', 'HasMissedFlight', 'HasHotelChange', 'HasModeChange', 'HasItineraryEdit',
    'DaysPreapproved', 'ExtensionLength'
]

print(f'Using {len(CATEGORICAL_COLS)} categorical features and {len(NUMERIC_COLS)} numeric features.')
print(f'Explicitly excluded (leakage risk): {LEAKAGE_COLS}')

Using 8 categorical features and 19 numeric features.
Explicitly excluded (leakage risk): ['Departure_CO2e', 'Return_CO2e', 'Hotel_CO2e', 'Spend_CO2e', 'TotalCO2e', 'HighCarbon']


In [14]:
y = train_df['HighCarbon']

X = train_df[CATEGORICAL_COLS + NUMERIC_COLS].copy()
X_test = test_df[CATEGORICAL_COLS + NUMERIC_COLS].copy()

for col in ['LeadTimeDays', 'ApprovalLeadDays']:
    median_val = X[col].median()
    X[col] = X[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[CATEGORICAL_COLS] = encoder.fit_transform(X[CATEGORICAL_COLS])
X_test[CATEGORICAL_COLS] = encoder.transform(X_test[CATEGORICAL_COLS])

print(X.shape, X_test.shape)
X.head(3)

(65289, 27) (21764, 27)


,DepartureLocationCountry,DepartureLocationCity,ArrivalLocationCountry,ArrivalLocationCity,ShippingTypeDescription,Purpose,OutOfPolicy,BusinessUnit,ShippingType,EntitiyCode,...,HasTripExtension,HasFlightChange,HasFlightCancellation,HasFlightDelay,HasMissedFlight,HasHotelChange,HasModeChange,HasItineraryEdit,DaysPreapproved,ExtensionLength
0,2.0,0.0,8.0,17.0,2.0,1.0,0.0,11.0,12,9000,...,0,0,0,0,0,0,0,0,2,0.0
1,7.0,7.0,11.0,14.0,5.0,1.0,1.0,10.0,11,6000,...,0,0,0,0,0,0,0,0,2,0.0
2,1.0,9.0,19.0,8.0,3.0,0.0,0.0,11.0,10,9000,...,0,0,0,0,0,0,0,0,1,0.0


# Train-test split

In [15]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=RANDOM_STATE
)
model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_probs)
print(f'Validation ROC-AUC: {auc:.4f}')

Validation ROC-AUC: 0.9994


In [16]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(10)

DepartureLocationCountry    0.220550
ArrivalLocationCountry      0.169897
ShippingTypeDescription     0.154536
ArrivalLocationCity         0.149802
DepartureLocationCity       0.090127
ShippingType                0.085849
OutOfPolicy                 0.042019
HotelNights                 0.038975
LeadTimeDays                0.003194
HasMissedFlight             0.003066
dtype: float32

In [17]:
final_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=RANDOM_STATE
)
final_model.fit(X, y)

test_probs = final_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TripID': test_df['TripID'],
    'HighCarbon': test_probs
})

print(submission.shape)
print(submission['TripID'].is_unique)
submission.head()

(21764, 2)
True


,TripID,HighCarbon
0,2,0.999845
1,4,0.006202
2,5,0.899262
3,14,0.000189
4,15,0.000251


# Submission file

In [18]:
submission.to_csv('predictions.csv', index=False)
print('Saved predictions.csv')
submission['HighCarbon'].describe()

Saved predictions.csv


count    21764.000000
mean         0.248453
std          0.423647
min          0.000002
25%          0.000177
50%          0.000560
75%          0.142836
max          0.999979
Name: HighCarbon, dtype: float64

## Summary

- **Model:** XGBoost binary classifier
- **Validation ROC-AUC:** see Step 5 output above
- **Features used:** route (departure/arrival country and city), transport type, trip purpose, business unit, policy status, hotel nights, cost, and a set of features engineered from the event log timestamps (lead time before departure, approval lead time, number of process events, and flags for things like manager preapproval, trip extensions, and flight or hotel changes)
- **Features explicitly excluded:** all CO2/emissions columns and the `HighCarbon` label itself, employee number
- **Output:** `predictions.csv`, containing a probability for every trip in the hidden test set, ready to submit alongside this notebook